In [ ]:
# Scores speech on validation sentences to show where quality is lost.
# Use GPU T4 x2. Add four inputs: prep-english, iiit-h, train-iith-eng, train-iith-tel.
# QUICK_TEST = True runs a few clips in a few minutes to prove the setup first.
QUICK_TEST = False
LANGUAGES = ["English", "Telugu"]
CLIPS = 6 if QUICK_TEST else 60
SWEEP_SPEAKERS = 2 if QUICK_TEST else 12
SWEEP_CLIPS = 3 if QUICK_TEST else 30

In [ ]:
import glob, os, shutil, subprocess, time
REPO = "https://github.com/23f2001033/IITH-Intern-Assignment-.git"
PROJECT = "/tmp/project"
shutil.rmtree(PROJECT, ignore_errors=True)
subprocess.run(["git", "clone", "--depth", "1", REPO, PROJECT], check=True)
print(subprocess.run(["git", "-C", PROJECT, "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout)
subprocess.run("pip install -q pyworld tgt g2p-en inflect unidecode", shell=True, check=True)

In [ ]:
# Unpack the features and find both checkpoints for each language
subprocess.run("nvidia-smi -L", shell=True)
subprocess.run("find /kaggle/input -maxdepth 6 -not -name '*.npy' | head -60", shell=True)

def checkpoints(language):
    best = glob.glob("/kaggle/input/**/{}/ckpt/best.pth.tar".format(language), recursive=True)
    assert len(best) == 1, "Expected one {} checkpoint folder, found: {}".format(language, best)
    folder = os.path.dirname(best[0])
    steps = [name.split(".")[0] for name in os.listdir(folder) if name[0].isdigit()]
    last = max(steps, key=int)
    return ["best=" + best[0], "last={}/{}.pth.tar".format(folder, last)]

for language in LANGUAGES:
    work = "/tmp/tts/" + language
    os.makedirs(work, exist_ok=True)
    tars = glob.glob("/kaggle/input/**/{}_preprocessed.tar*".format(language), recursive=True)
    tars = [path for path in tars if os.path.isfile(path)]
    assert len(tars) == 1, "Expected one {} features tar, found: {}".format(language, tars)
    subprocess.run(["tar", "-xf", tars[0], "-C", work], check=True)
    print(language, checkpoints(language))

In [ ]:
# Score both languages at once, one per GPU
os.makedirs("/kaggle/working/logs", exist_ok=True)
jobs = {}
for gpu, language in enumerate(LANGUAGES):
    command = ["python", PROJECT + "/scripts/evaluate_tts.py", "--language", language,
               "--work_dir", "/tmp/tts/" + language,
               "--results_dir", "/kaggle/working/evaluation/" + language,
               "--audio_dir", "/tmp/eval_audio/" + language, "--clips", str(CLIPS)]
    for item in checkpoints(language):
        command += ["--ckpt", item]
    if language == "English":
        command += ["--sweep_speakers", str(SWEEP_SPEAKERS), "--sweep_clips", str(SWEEP_CLIPS)]
    log = open("/kaggle/working/logs/evaluate_{}.log".format(language), "a")
    jobs[language] = subprocess.Popen(command, stdout=log, stderr=subprocess.STDOUT,
                                      env=dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu)))

def last_line(language):
    lines = open("/kaggle/working/logs/evaluate_{}.log".format(language), errors="replace").readlines()
    return lines[-1].strip()[:150] if lines else "(starting)"

while any(p.poll() is None for p in jobs.values()):
    time.sleep(60)
    for language in jobs:
        print(time.strftime("%H:%M"), language, last_line(language), flush=True)
print({k: p.returncode for k, p in jobs.items()})
for language in jobs:
    print(language, open("/kaggle/working/logs/evaluate_{}.log".format(language), errors="replace").read()[-6000:])
assert all(p.returncode == 0 for p in jobs.values()), "A scoring job failed; see the log above"